# PKG Attrition — Staging Payments & Deposit Table EDA

**PNC Treasury Management · Data Science** — read-only diagnostic notebook.

Profiles `neo4j_payments` (staging transactions) and the deposit daily-balance
table over **2024-01-01 → 2026-07-31**, to settle the definitional questions
that gate the attrition study.

**No writes.** Temp views and `persist()` on narrow frames only. Nothing is
written to Hive, HDFS or local disk.

**Output discipline.** Ten sections, **one consolidated output block each**, so
the whole run is ten screenshots. Everything renders as pandas, never
`.show()`. §10 reprints every answer in a single scoreboard.

| § | Question it settles |
|---|---|
| 0 | Schemas resolve; do sibling `avg_monthly_bal_N` columns exist? |
| 1 | Table shapes, date coverage, `trans_id` uniqueness |
| 2 | `deposit_family` inventory → the DDA/MMDA filter |
| 3 | Deposit grain: one row per account per business day? |
| 4 | Are `acct_status` / `closed_dt` **as-of** or **current-state**? |
| 5 | What is `avg_monthly_bal_1`? Monthly panel construction |
| 6 | Closure definitions vs. the actual flag |
| 7 | Payments profile, direction split, duplicate-leg check |
| 8 | Payments ↔ deposits join coverage; `mdm_id` ↔ `cust_pwr_id` |
| 9 | Attrition feasibility: closures per month, decline shape, episode count |

**Run order note.** §2 prints the family inventory using an auto-detect regex.
If the regex mis-classifies anything, set `DDA_MMDA_FAMILIES` explicitly in §0
and re-run from §2 down.

## §0 · Configuration, helpers, schema resolution

In [ ]:
# ---------------------------------------------------------------- CONFIG ----
PAYMENTS_TBL = "<db>.neo4j_payments"        # <<< CONFIRM
DEPOSITS_TBL = "<db>.<deposit_table>"       # <<< CONFIRM

DATE_START = "2024-01-01"
DATE_END   = "2026-07-31"

# Account-level sampling for the whole notebook. 100 = all accounts.
# Set to 2 for a fast smoke run over ~2% of accounts (deterministic by hash),
# then raise to 100 for the real pass. Sampling is on ACCOUNTS, not rows, so
# every per-account diagnostic stays internally consistent.
ACCT_SAMPLE_PCT = 100

# Smaller samples for the expensive relational diagnostics (§5 lag joins, §3
# duplicate row dumps). These are always sampled regardless of ACCT_SAMPLE_PCT.
AMB_SAMPLE_PCT     = 5          # % of accounts used for the avg_monthly_bal_1 test
DUP_AXIS_KEYS      = 2_000      # duplicate (acct, date) keys for the split-axis
DUP_ROW_EXAMPLES   = 4          # duplicate keys dumped with all columns

# Deposit family scope. None = auto-detect with the regex below.
# After reading the §2 inventory, replace with an explicit list, e.g.
#   DDA_MMDA_FAMILIES = ["COMMERCIAL DDA", "MMDA", "BUSINESS CHECKING"]
DDA_MMDA_FAMILIES = None
DDA_MMDA_REGEX    = r"(?i)(DDA|DEMAND|CHECK|MMDA|MONEY\s*MARKET|\bMM\b|NOW\b)"

# Closure-definition parameters (§6)
GONE_DAYS   = 45        # no deposit row in the last N days of the window
ZERO_TOL    = 100.0     # |balance| <= this counts as "drained"
MIN_BASE_BAL = 25_000.0 # brief §6.2 small-balance floor, for the 30% rule

CHECK_TRANS_ID_UNIQUE = True   # one extra shuffle over payments; worth it once

# ---------------------------------------------------------------- IMPORTS ---
import re
import pandas as pd
from pyspark.sql import functions as F, Window as W
from pyspark.storagelevel import StorageLevel
from IPython.display import display, Markdown

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 200)
pd.set_option("display.width", 220)
pd.set_option("display.max_colwidth", 48)
pd.set_option("display.float_format", lambda v: f"{v:,.4f}".rstrip("0").rstrip(".")
              if abs(v) < 1e4 else f"{v:,.1f}")

ANSWERS = {}


def ans(key, value):
    """Record a headline answer for the §10 scoreboard."""
    ANSWERS[key] = value
    return value


def H(title, sub=None):
    display(Markdown(f"### {title}" + (f"\n\n_{sub}_" if sub else "")))


def T(pdf, title=None):
    if title:
        display(Markdown(f"**{title}**"))
    display(pdf)


def kv(d, name="value"):
    """dict -> one-column pandas frame, for compact metric blocks."""
    return pd.DataFrame({name: pd.Series(d, dtype=object)})


def as_date(c):
    """Parse a date column that may arrive as date, timestamp, 'YYYY-MM-DD',
    or an 8-digit yyyyMMdd integer/string. Returns a DateType column."""
    col = F.col(c) if isinstance(c, str) else c
    return F.coalesce(
        F.to_date(col),
        F.to_date(col.cast("string"), "yyyy-MM-dd"),
        F.to_date(col.cast("string"), "yyyyMMdd"),
    )


def nz(c):
    """Empty string and whitespace are absent, same as NULL. Applied to every
    string key — a '' mdm_id that reads as populated would mis-type the
    direction of every leg it touches."""
    col = F.col(c) if isinstance(c, str) else c
    return F.when(F.trim(col.cast("string")) == "", None).otherwise(F.trim(col.cast("string")))


def nulltok(c):
    """countDistinct silently drops NULL. Coalescing to a visible token makes
    'this column is NULL on one row and populated on the other' countable."""
    col = F.col(c) if isinstance(c, str) else c
    return F.coalesce(col.cast("string"), F.lit("<NULL>"))


def bucket(c, n=100):
    return F.pmod(F.hash(F.col(c)), F.lit(n))


def resolve(df, wanted):
    """Case-insensitive column resolution. Fail-fast on anything missing."""
    lut = {c.lower(): c for c in df.columns}
    found, missing = {}, []
    for w in wanted:
        (found.__setitem__(w, lut[w.lower()]) if w.lower() in lut else missing.append(w))
    return found, missing


# ---------------------------------------------------------------- RESOLVE ---
PAY_COLS = ["trans_id", "trans_dt", "trans_amt", "payment_rail", "category",
            "mdm_id_pays", "customer_name_pays", "mdm_id_receives", "customer_name_receives",
            "pnc_dep_acct_pays", "pnc_dep_acct_receives",
            "unq_cpty_acct_id", "cpty_name", "cpty_fin_entity_name"]

DEP_COLS = ["acct_full_acct_id", "edw_tda_load_dt", "balance", "acct_status",
            "avg_monthly_bal_1", "cust_pwr_id", "cust_name",
            "opened_dt", "closed_dt", "deposit_family"]

pay_raw = spark.table(PAYMENTS_TBL)
dep_raw = spark.table(DEPOSITS_TBL)

pay_found, pay_missing = resolve(pay_raw, PAY_COLS)
dep_found, dep_missing = resolve(dep_raw, DEP_COLS)

if pay_missing or dep_missing:
    raise KeyError(f"Missing columns — payments: {pay_missing} | deposits: {dep_missing}. "
                   f"Fix the name map in §0 before continuing.")

# Sibling avg_monthly_bal_N columns are strong evidence that _1 means
# "one month back" rather than month-to-date.
amb_siblings = sorted(c for c in dep_raw.columns
                      if re.fullmatch(r"(?i)avg_monthly_bal_\d+", c))

schema_pdf = pd.concat([
    pd.DataFrame([(f.name, str(f.dataType), "payments") for f in pay_raw.schema.fields
                  if f.name in pay_found.values()],
                 columns=["column", "spark_type", "table"]),
    pd.DataFrame([(f.name, str(f.dataType), "deposits") for f in dep_raw.schema.fields
                  if f.name in dep_found.values()],
                 columns=["column", "spark_type", "table"]),
], ignore_index=True)

H("§0 · Schema resolution")
T(schema_pdf, "Resolved columns and source dtypes")
T(kv({
    "payments table": PAYMENTS_TBL,
    "deposits table": DEPOSITS_TBL,
    "window": f"{DATE_START} .. {DATE_END}",
    "account sample %": ACCT_SAMPLE_PCT,
    "avg_monthly_bal_* columns present": ", ".join(amb_siblings) or "(only _1)",
    "-> sibling columns imply": ("_N = N months back (prior complete months)"
                                 if len(amb_siblings) > 1
                                 else "inconclusive from names alone — tested in §5"),
}), "Configuration")

ans("amb_sibling_columns", ", ".join(amb_siblings))

## §1 · Table shapes and date coverage

In [ ]:
# Canonical projections. Column pruning happens here, before any scan.
pay = (pay_raw
       .select(
           nz(pay_found["trans_id"]).alias("trans_id"),
           as_date(pay_found["trans_dt"]).alias("trans_dt"),
           F.col(pay_found["trans_amt"]).cast("double").alias("trans_amt"),
           nz(pay_found["payment_rail"]).alias("rail"),
           nz(pay_found["category"]).alias("category"),
           nz(pay_found["mdm_id_pays"]).alias("mdm_pays"),
           nz(pay_found["customer_name_pays"]).alias("name_pays"),
           nz(pay_found["mdm_id_receives"]).alias("mdm_recv"),
           nz(pay_found["customer_name_receives"]).alias("name_recv"),
           nz(pay_found["pnc_dep_acct_pays"]).alias("acct_pays"),
           nz(pay_found["pnc_dep_acct_receives"]).alias("acct_recv"),
           nz(pay_found["unq_cpty_acct_id"]).alias("cpty_acct"),
           nz(pay_found["cpty_name"]).alias("cpty_name"),
           nz(pay_found["cpty_fin_entity_name"]).alias("cpty_fi"),
       )
       .filter(F.col("trans_dt").between(F.lit(DATE_START), F.lit(DATE_END))))

dep_all = (dep_raw
           .select(
               nz(dep_found["acct_full_acct_id"]).alias("acct"),
               as_date(dep_found["edw_tda_load_dt"]).alias("dt"),
               F.col(dep_found["balance"]).cast("double").alias("balance"),
               nz(dep_found["acct_status"]).alias("status"),
               F.col(dep_found["avg_monthly_bal_1"]).cast("double").alias("amb1"),
               nz(dep_found["cust_pwr_id"]).alias("cust_pwr_id"),
               nz(dep_found["cust_name"]).alias("cust_name"),
               as_date(dep_found["opened_dt"]).alias("opened_dt"),
               as_date(dep_found["closed_dt"]).alias("closed_dt"),
               nz(dep_found["deposit_family"]).alias("family"),
           )
           .filter(F.col("dt").between(F.lit(DATE_START), F.lit(DATE_END))))

if ACCT_SAMPLE_PCT < 100:
    dep_all = dep_all.filter(bucket("acct") < ACCT_SAMPLE_PCT)

pay.createOrReplaceTempView("v_pay")
dep_all.createOrReplaceTempView("v_dep_all")

pay_shape = pay.agg(
    F.count(F.lit(1)).alias("rows"),
    F.approx_count_distinct("trans_id", 0.01).alias("trans_id_dist_approx"),
    F.min("trans_dt").alias("min_dt"), F.max("trans_dt").alias("max_dt"),
    F.countDistinct(F.date_format("trans_dt", "yyyy-MM")).alias("n_months"),
    F.approx_count_distinct("mdm_pays", 0.01).alias("mdm_pays_dist"),
    F.approx_count_distinct("mdm_recv", 0.01).alias("mdm_recv_dist"),
    F.approx_count_distinct("cpty_acct", 0.01).alias("cpty_acct_dist"),
    F.sum("trans_amt").alias("total_amt"),
).toPandas().iloc[0]

dep_shape = dep_all.agg(
    F.count(F.lit(1)).alias("rows"),
    F.approx_count_distinct("acct", 0.01).alias("acct_dist"),
    F.approx_count_distinct("cust_pwr_id", 0.01).alias("cust_pwr_dist"),
    F.min("dt").alias("min_dt"), F.max("dt").alias("max_dt"),
    F.countDistinct("dt").alias("n_distinct_dates"),
    F.countDistinct(F.date_format("dt", "yyyy-MM")).alias("n_months"),
).toPandas().iloc[0]

# Weekday coverage — do weekend rows exist, or is this a business-day series?
dow = (dep_all.groupBy(F.date_format("dt", "E").alias("weekday"))
       .agg(F.countDistinct("dt").alias("n_dates"), F.count(F.lit(1)).alias("rows"))
       .toPandas().sort_values("rows", ascending=False))

dup_tid = None
if CHECK_TRANS_ID_UNIQUE:
    dup_tid = (pay.groupBy("trans_id").count().filter("count > 1")
               .agg(F.count(F.lit(1)).alias("dup_keys"),
                    F.sum("count").alias("dup_rows"),
                    F.max("count").alias("max_rows_per_id"))
               .toPandas().iloc[0])

H("§1 · Table shapes and date coverage")
T(pd.concat([
    kv({f"payments · {k}": v for k, v in pay_shape.items()}),
    kv({f"deposits · {k}": v for k, v in dep_shape.items()}),
]), "Shape")
T(dow, "Deposit rows by weekday — weekend rows here mean it is not a business-day series")
if dup_tid is not None:
    T(kv(dup_tid.to_dict()), "trans_id uniqueness (expect all zero)")
    ans("trans_id_unique", "YES" if dup_tid["dup_keys"] == 0
        else f"NO — {int(dup_tid['dup_keys']):,} ids on >1 row")

ans("payments_rows", f"{int(pay_shape['rows']):,}")
ans("deposit_rows", f"{int(dep_shape['rows']):,}")
ans("deposit_accounts_all_families", f"~{int(dep_shape['acct_dist']):,}")
ans("deposit_weekend_rows",
    "yes" if dow.loc[dow.weekday.isin(["Sat", "Sun"]), "rows"].sum() > 0 else "no (business days only)")

## §2 · `deposit_family` inventory → the DDA/MMDA scope

The auto-detect flag below is a *guess from the family name*. Read the table,
then set `DDA_MMDA_FAMILIES` explicitly in §0 if it is wrong.

In [ ]:
fam = (dep_all
       .groupBy("family")
       .agg(F.count(F.lit(1)).alias("rows"),
            F.approx_count_distinct("acct", 0.01).alias("accts_approx"),
            F.expr("percentile_approx(balance, 0.5)").alias("median_balance"),
            F.avg("balance").alias("mean_balance"),
            F.sum((F.col("balance") < 0).cast("int")).alias("neg_bal_rows"))
       .toPandas())

fam["auto_dda_mmda"] = fam["family"].fillna("").str.contains(DDA_MMDA_REGEX, regex=True)
fam["row_share"] = fam["rows"] / fam["rows"].sum()
fam = fam.sort_values("rows", ascending=False).reset_index(drop=True)

if DDA_MMDA_FAMILIES is None:
    SCOPE_FAMILIES = fam.loc[fam.auto_dda_mmda, "family"].dropna().tolist()
    scope_note = "AUTO-DETECTED from family name — confirm against the table above"
else:
    SCOPE_FAMILIES = list(DDA_MMDA_FAMILIES)
    scope_note = "EXPLICIT list from §0"

dep = dep_all.filter(F.col("family").isin(SCOPE_FAMILIES))
dep = dep.persist(StorageLevel.MEMORY_AND_DISK)
dep.createOrReplaceTempView("v_dep")

scope_shape = dep.agg(
    F.count(F.lit(1)).alias("rows"),
    F.approx_count_distinct("acct", 0.01).alias("accts"),
    F.approx_count_distinct("cust_pwr_id", 0.01).alias("customers"),
).toPandas().iloc[0]

H("§2 · Deposit family inventory")
T(fam[["family", "rows", "row_share", "accts_approx", "median_balance",
       "mean_balance", "neg_bal_rows", "auto_dda_mmda"]],
  "All families in the window")
T(kv({
    "scope selection": scope_note,
    "families in scope": ", ".join(map(str, SCOPE_FAMILIES))[:400],
    "n families in scope": len(SCOPE_FAMILIES),
    "rows in scope": f"{int(scope_shape['rows']):,}",
    "accounts in scope": f"~{int(scope_shape['accts']):,}",
    "customers in scope": f"~{int(scope_shape['customers']):,}",
    "share of all deposit rows": f"{scope_shape['rows'] / dep_shape['rows']:.1%}",
}), "DDA / MMDA scope applied from here down")

ans("dda_mmda_families", ", ".join(map(str, SCOPE_FAMILIES))[:200])
ans("dda_mmda_accounts", f"~{int(scope_shape['accts']):,}")

## §3 · Deposit grain — one row per account per business day?

Three outputs: how often the key repeats, **which column the repeats straddle**
(the splitting axis), and a full-column dump of a few repeated keys so the
difference is visible rather than inferred.

In [ ]:
key_ct = (dep.groupBy("acct", "dt").agg(F.count(F.lit(1)).alias("n"))
          .persist(StorageLevel.MEMORY_AND_DISK))

grain = key_ct.agg(
    F.count(F.lit(1)).alias("distinct_acct_date_keys"),
    F.sum("n").alias("total_rows"),
    F.sum((F.col("n") > 1).cast("int")).alias("keys_with_dupes"),
    F.max("n").alias("max_rows_per_key"),
).toPandas().iloc[0]

rows_per_key = (key_ct.groupBy("n").agg(F.count(F.lit(1)).alias("n_keys"))
                .orderBy("n").toPandas())
rows_per_key["share_of_keys"] = rows_per_key["n_keys"] / rows_per_key["n_keys"].sum()

# --- splitting axis: within a duplicated key, which columns actually differ? ---
axis_pdf = pd.DataFrame()
dup_dump = pd.DataFrame()
if grain["keys_with_dupes"] > 0:
    dup_keys = key_ct.filter("n > 1").select("acct", "dt").limit(DUP_AXIS_KEYS).persist()
    dr = dep.join(dup_keys, ["acct", "dt"], "inner")
    varcols = [c for c in dep.columns if c not in ("acct", "dt")]
    nd = dr.groupBy("acct", "dt").agg(
        *[F.countDistinct(nulltok(c)).alias(c) for c in varcols])
    axis = nd.agg(*[F.sum((F.col(c) > 1).cast("int")).alias(c) for c in varcols],
                  F.count(F.lit(1)).alias("_groups")).toPandas().iloc[0]
    ngroups = int(axis.pop("_groups"))
    axis_pdf = (pd.DataFrame({"groups_where_column_differs": axis.astype(int)})
                .assign(share_of_dup_groups=lambda d: d.iloc[:, 0] / ngroups)
                .sort_values("groups_where_column_differs", ascending=False))
    dump_keys = dup_keys.limit(DUP_ROW_EXAMPLES)
    dup_dump = (dep.join(dump_keys, ["acct", "dt"], "inner")
                .orderBy("acct", "dt").limit(DUP_ROW_EXAMPLES * 6).toPandas())
    dup_keys.unpersist()

# --- rows per account-month vs. business days available -----------------------
per_am = (dep.groupBy("acct", F.date_format("dt", "yyyy-MM").alias("month"))
          .agg(F.countDistinct("dt").alias("n_days")))
bizdays = (dep.groupBy(F.date_format("dt", "yyyy-MM").alias("month"))
           .agg(F.countDistinct("dt").alias("bizdays_in_month")))
cov = (per_am.join(bizdays, "month")
       .withColumn("day_cov", F.col("n_days") / F.col("bizdays_in_month"))
       .agg(F.expr("percentile_approx(day_cov, array(0.01,0.10,0.25,0.50,0.75,0.90,1.0))")
            .alias("p"),
            F.avg((F.col("day_cov") >= 0.99).cast("double")).alias("share_full_month"))
       .toPandas().iloc[0])

H("§3 · Deposit grain")
T(kv({
    "distinct (acct, date) keys": f"{int(grain['distinct_acct_date_keys']):,}",
    "total rows": f"{int(grain['total_rows']):,}",
    "keys appearing more than once": f"{int(grain['keys_with_dupes']):,}",
    "share of keys duplicated": f"{grain['keys_with_dupes'] / grain['distinct_acct_date_keys']:.4%}",
    "max rows on one key": int(grain["max_rows_per_key"]),
}), "Key repetition")
T(rows_per_key, "Rows per (account, date)")
if not axis_pdf.empty:
    T(axis_pdf, "SPLITTING AXIS — which column the duplicate rows differ on "
                f"(sampled {DUP_AXIS_KEYS:,} duplicated keys)")
    T(dup_dump, "Raw duplicate rows, all columns")
T(pd.DataFrame({"day_coverage_pctile": ["p01", "p10", "p25", "p50", "p75", "p90", "max"],
                "value": cov["p"]}),
  "Per account-month: observed days / business days in that month")
T(kv({"account-months with ~full day coverage": f"{cov['share_full_month']:.1%}"}))

ans("deposit_grain",
    "one row per (acct, business day)" if grain["keys_with_dupes"] == 0
    else f"NOT unique — {grain['keys_with_dupes'] / grain['distinct_acct_date_keys']:.3%} of keys repeat "
         f"(splitting axis in §3)")
ans("acct_month_full_day_coverage", f"{cov['share_full_month']:.1%}")

## §4 · Are `acct_status` and `closed_dt` as-of, or current-state?

**The decisive test.** If an account that closed in 2025-06 already reads `'C'`
on its 2024-03 row, the column is a current-state stamp: usable as an *outcome
label* only, and never as a feature or an as-of state. The same test runs on
`closed_dt`.

In [ ]:
status_vals = (dep.groupBy(F.coalesce("status", F.lit("<NULL>")).alias("status"))
               .agg(F.count(F.lit(1)).alias("rows"),
                    F.approx_count_distinct("acct", 0.01).alias("accts_approx"))
               .orderBy(F.desc("rows")).toPandas())
status_vals["row_share"] = status_vals["rows"] / status_vals["rows"].sum()

# One row per account: does status / closed_dt move within an account?
acct = (dep.groupBy("acct").agg(
    F.min("dt").alias("first_dt"),
    F.max("dt").alias("last_dt"),
    F.countDistinct("dt").alias("n_days"),
    F.countDistinct(F.date_format("dt", "yyyy-MM")).alias("n_months"),
    F.countDistinct(nulltok("status")).alias("n_status_values"),
    F.countDistinct(nulltok("closed_dt")).alias("n_closed_dt_values"),
    F.expr("min_by(status, dt)").alias("status_first"),
    F.expr("max_by(status, dt)").alias("status_last"),
    F.expr("max_by(balance, dt)").alias("bal_last"),
    F.min("closed_dt").alias("closed_dt_min"),
    F.max("closed_dt").alias("closed_dt_max"),
    F.min("opened_dt").alias("opened_dt"),
    F.first("cust_pwr_id", ignorenulls=True).alias("cust_pwr_id"),
    F.first("family", ignorenulls=True).alias("family"),
).persist(StorageLevel.MEMORY_AND_DISK))

stability = acct.agg(
    F.count(F.lit(1)).alias("accounts"),
    F.avg((F.col("n_status_values") == 1).cast("double")).alias("share_status_constant"),
    F.avg((F.col("n_closed_dt_values") == 1).cast("double")).alias("share_closed_dt_constant"),
    F.avg(F.col("closed_dt_max").isNotNull().cast("double")).alias("share_with_closed_dt"),
    F.avg((F.col("status_last") == "C").cast("double")).alias("share_last_status_C"),
    F.avg((F.col("status_first") == "C").cast("double")).alias("share_first_status_C"),
).toPandas().iloc[0]

trans = (acct.groupBy(F.coalesce("status_first", F.lit("<NULL>")).alias("first"),
                      F.coalesce("status_last", F.lit("<NULL>")).alias("last"))
         .count().orderBy(F.desc("count")).limit(20).toPandas())

# THE test: on rows dated strictly before closed_dt, is the row already 'C'?
pre_close = (dep.filter(F.col("closed_dt").isNotNull() & (F.col("dt") < F.col("closed_dt")))
             .agg(F.count(F.lit(1)).alias("rows_before_closed_dt"),
                  F.avg((F.col("status") == "C").cast("double")).alias("share_already_C"))
             .toPandas().iloc[0])

# And: do rows continue to appear after closed_dt?
post_close = (dep.filter(F.col("closed_dt").isNotNull() & (F.col("dt") > F.col("closed_dt")))
              .agg(F.count(F.lit(1)).alias("rows_after_closed_dt"),
                   F.approx_count_distinct("acct", 0.01).alias("accts_with_post_rows"),
                   F.expr("percentile_approx(datediff(dt, closed_dt), array(0.5,0.9,1.0))")
                   .alias("days_after_p50_p90_max"),
                   F.avg((F.abs(F.col("balance")) <= ZERO_TOL).cast("double"))
                   .alias("share_post_rows_zero_bal"))
              .toPandas().iloc[0])

n_closed_accts = acct.filter(F.col("closed_dt_max").isNotNull()).count()

# Opened_dt vs first observed row
open_chk = (acct.filter(F.col("opened_dt").isNotNull())
            .agg(F.expr("percentile_approx(datediff(first_dt, opened_dt), array(0.05,0.5,0.95))")
                 .alias("p"),
                 F.avg((F.col("opened_dt") >= F.lit(DATE_START)).cast("double"))
                 .alias("share_opened_in_window"))
            .toPandas().iloc[0])

status_is_current_state = bool(pre_close["share_already_C"] is not None
                               and pre_close["share_already_C"] > 0.5)

H("§4 · Account status and closure-date semantics")
T(status_vals, "acct_status values")
T(kv({
    "accounts in scope": f"{int(stability['accounts']):,}",
    "status constant across the account's life": f"{stability['share_status_constant']:.1%}",
    "closed_dt constant across the account's life": f"{stability['share_closed_dt_constant']:.1%}",
    "accounts with any closed_dt": f"{stability['share_with_closed_dt']:.1%} "
                                   f"({n_closed_accts:,} accounts)",
    "status = 'C' on LAST row": f"{stability['share_last_status_C']:.1%}",
    "status = 'C' on FIRST row": f"{stability['share_first_status_C']:.1%}",
}), "Within-account stability")
T(trans, "status_first x status_last (top 20)")
T(kv({
    "rows dated BEFORE closed_dt": f"{int(pre_close['rows_before_closed_dt'] or 0):,}",
    ">>> of those, share already showing 'C'": f"{(pre_close['share_already_C'] or 0):.1%}",
    "VERDICT": ("CURRENT-STATE — status is stamped on history. Outcome label only, "
                "never a feature or an as-of state."
                if status_is_current_state else
                "AS-OF — status reflects the row's own date. Usable as a time-varying state."),
}), "*** The decisive test ***")
T(kv({
    "rows dated AFTER closed_dt": f"{int(post_close['rows_after_closed_dt'] or 0):,}",
    "accounts still reporting after closed_dt": f"~{int(post_close['accts_with_post_rows'] or 0):,}",
    "days after closure (p50 / p90 / max)": post_close["days_after_p50_p90_max"],
    "share of post-closure rows at ~zero balance": f"{(post_close['share_post_rows_zero_bal'] or 0):.1%}",
}), "Do rows continue after closed_dt?")
T(kv({
    "first_dt - opened_dt (p05 / p50 / p95, days)": open_chk["p"],
    "accounts opened inside the window": f"{open_chk['share_opened_in_window']:.1%}",
}), "opened_dt vs first observed row")

ans("status_semantics", "CURRENT-STATE (label only)" if status_is_current_state else "AS-OF (usable as state)")
ans("rows_after_closed_dt", f"{int(post_close['rows_after_closed_dt'] or 0):,} rows / "
                           f"~{int(post_close['accts_with_post_rows'] or 0):,} accounts")
ans("accounts_with_closed_dt", f"{n_closed_accts:,} ({stability['share_with_closed_dt']:.1%})")

## §5 · What is `avg_monthly_bal_1`? And the monthly panel

Four hypotheses, one clean discriminator first: **how many distinct values does
`avg_monthly_bal_1` take within one account-month?**

- exactly 1 → it is a *prior-period* figure (H2 prior-month mean, or H4 prior month-end)
- ≈ number of days → it is *running within the month* (H1 MTD, or H3 trailing-30d)

Then each hypothesis is scored by median absolute relative difference.

In [ ]:
depm = dep.withColumn("month", F.date_format("dt", "yyyy-MM"))

mp = (depm.groupBy("acct", "month").agg(
    F.count(F.lit(1)).alias("n_rows"),
    F.countDistinct("dt").alias("n_days"),
    F.max("dt").alias("last_dt"),
    F.expr("max_by(balance, dt)").alias("bal_eom"),
    F.expr("min_by(balance, dt)").alias("bal_som"),
    F.avg("balance").alias("bal_mean"),
    F.min("balance").alias("bal_min"),
    F.max("balance").alias("bal_max"),
    F.expr("percentile_approx(balance, 0.5)").alias("bal_med"),
    F.expr("max_by(amb1, dt)").alias("amb_eom"),
    F.expr("min_by(amb1, dt)").alias("amb_som"),
    F.countDistinct(nulltok("amb1")).alias("n_amb_distinct"),
    F.expr("max_by(status, dt)").alias("status_eom"),
    F.max("closed_dt").alias("closed_dt"),
    F.first("cust_pwr_id", ignorenulls=True).alias("cust_pwr_id"),
    F.first("family", ignorenulls=True).alias("family"),
).withColumn("month_idx",
             (F.substring("month", 1, 4).cast("int") * 12
              + F.substring("month", 6, 2).cast("int")))
      .persist(StorageLevel.MEMORY_AND_DISK))
mp.createOrReplaceTempView("v_monthly")

variability = (mp.filter(F.col("n_days") >= 15)
               .withColumn("amb_moves", (F.col("n_amb_distinct") > 1).cast("double"))
               .agg(F.avg("amb_moves").alias("share_amb_varies_within_month"),
                    F.expr("percentile_approx(n_amb_distinct, array(0.5,0.9))").alias("p"),
                    F.expr("percentile_approx(n_days, 0.5)").alias("median_days"),
                    F.avg(F.col("amb_eom").isNull().cast("double")).alias("share_amb_null"))
               .toPandas().iloc[0])

# Hypothesis scoring on a deterministic account sample
samp = mp.filter(bucket("acct", 100) < AMB_SAMPLE_PCT)
wa = W.partitionBy("acct").orderBy("month_idx")
samp = (samp
        .withColumn("prev_idx", F.lag("month_idx").over(wa))
        .withColumn("prev_bal_mean", F.lag("bal_mean").over(wa))
        .withColumn("prev_bal_eom", F.lag("bal_eom").over(wa))
        .filter(F.col("month_idx") - F.col("prev_idx") == 1))


def _rel(a, b):
    return F.when(F.abs(F.col(b)) > 1.0,
                  F.abs(F.col(a) - F.col(b)) / F.abs(F.col(b)))


hyp = {
    "H1  amb(eom) == mean of THIS month's daily balance":   ("amb_eom", "bal_mean"),
    "H2  amb(any) == mean of PRIOR month's daily balance":  ("amb_som", "prev_bal_mean"),
    "H3  amb(eom) == PRIOR month's daily mean (eom read)":  ("amb_eom", "prev_bal_mean"),
    "H4  amb(any) == PRIOR month-end balance":              ("amb_som", "prev_bal_eom"),
}
rows = []
for label, (a, b) in hyp.items():
    r = (samp.filter(F.col(a).isNotNull() & F.col(b).isNotNull())
         .withColumn("rd", _rel(a, b)).filter(F.col("rd").isNotNull())
         .agg(F.count(F.lit(1)).alias("n"),
              F.expr("percentile_approx(rd, 0.5)").alias("median_abs_rel_diff"),
              F.avg((F.col("rd") <= 0.005).cast("double")).alias("share_within_0p5pct"),
              F.avg((F.col("rd") <= 0.05).cast("double")).alias("share_within_5pct"))
         .toPandas().iloc[0])
    rows.append({"hypothesis": label, **r.to_dict()})
hyp_pdf = pd.DataFrame(rows).sort_values("share_within_0p5pct", ascending=False)

bal_dist = (mp.agg(
    F.avg((F.col("bal_eom") < 0).cast("double")).alias("share_negative_eom"),
    F.avg((F.abs(F.col("bal_eom")) <= ZERO_TOL).cast("double")).alias("share_zeroish_eom"),
    F.expr("percentile_approx(bal_eom, array(0.01,0.10,0.25,0.50,0.75,0.90,0.99))").alias("p"),
).toPandas().iloc[0])

winner = hyp_pdf.iloc[0]

H("§5 · avg_monthly_bal_1 semantics, and the monthly panel")
T(kv({
    "account-months (>=15 days) where amb1 VARIES within the month":
        f"{variability['share_amb_varies_within_month']:.1%}",
    "distinct amb1 values per account-month (p50 / p90)": variability["p"],
    "median observed days per account-month": variability["median_days"],
    "amb1 NULL share": f"{variability['share_amb_null']:.1%}",
    ">>> reading": ("RUNNING within the month -> month-to-date or trailing window"
                    if variability["share_amb_varies_within_month"] > 0.5
                    else "CONSTANT within the month -> a PRIOR-period figure"),
}), "Discriminator: does amb1 move within a month?")
T(hyp_pdf, f"Hypothesis scoring (~{AMB_SAMPLE_PCT}% of accounts, consecutive month pairs)")
T(kv({"BEST FIT": winner["hypothesis"],
      "matched within 0.5%": f"{winner['share_within_0p5pct']:.1%}",
      "median abs rel diff": f"{winner['median_abs_rel_diff']:.4f}"}))
T(pd.DataFrame({"pctile": ["p01", "p10", "p25", "p50", "p75", "p90", "p99"],
                "bal_eom": bal_dist["p"]}), "Month-end balance distribution")
T(kv({"account-months with negative month-end balance": f"{bal_dist['share_negative_eom']:.2%}",
      "account-months at ~zero month-end balance": f"{bal_dist['share_zeroish_eom']:.2%}"}))

ans("avg_monthly_bal_1_meaning",
    f"{winner['hypothesis']} ({winner['share_within_0p5pct']:.0%} within 0.5%)")
ans("negative_balance_share", f"{bal_dist['share_negative_eom']:.2%} of account-months")

## §6 · Closure definitions vs. the actual flag

Five definitions scored against each other. The flag is the reference, but it
is only usable as a label if §4 says it is as-of — and even then, an account
that goes silent without ever being flagged is still a departure.

In [ ]:
WIN_END = pd.Timestamp(DATE_END)
gone_cut = (WIN_END - pd.Timedelta(days=GONE_DAYS)).strftime("%Y-%m-%d")

wa2 = W.partitionBy("acct").orderBy("month_idx")
w_last3 = wa2.rowsBetween(-2, 0)
w_tr3 = wa2.rowsBetween(-2, 0)
w_pr6 = wa2.rowsBetween(-8, -3)

mpr = (mp
       .withColumn("bal3", F.avg("bal_eom").over(w_last3))
       .withColumn("trail3", F.avg("bal_eom").over(w_tr3))
       .withColumn("prior6", F.avg("bal_eom").over(w_pr6))
       .withColumn("n_prior6", F.count("bal_eom").over(w_pr6))
       .withColumn("rule30",
                   ((F.col("n_prior6") >= 6) & (F.col("prior6") > MIN_BASE_BAL)
                    & (F.col("trail3") <= 0.70 * F.col("prior6"))).cast("int")))

rule_hits = (mpr.groupBy("acct")
             .agg(F.max("rule30").alias("d5_rule30_ever"),
                  F.min(F.when(F.col("rule30") == 1, F.col("month"))).alias("d5_first_month"),
                  F.expr("max_by(bal3, month_idx)").alias("bal3_last")))

defs = (acct.join(rule_hits, "acct", "left")
        .withColumn("d1_status_C", (F.col("status_last") == "C").cast("int"))
        .withColumn("d2_closed_dt", F.col("closed_dt_max").isNotNull().cast("int"))
        .withColumn("d3_went_silent", (F.col("last_dt") < F.lit(gone_cut)).cast("int"))
        .withColumn("d4_drained_and_silent",
                    ((F.col("last_dt") < F.lit(gone_cut))
                     & (F.abs(F.coalesce(F.col("bal3_last"), F.lit(0.0))) <= ZERO_TOL)).cast("int"))
        .withColumn("d5_rule30_ever", F.coalesce(F.col("d5_rule30_ever"), F.lit(0)))
        .persist(StorageLevel.MEMORY_AND_DISK))

DEFS = ["d1_status_C", "d2_closed_dt", "d3_went_silent", "d4_drained_and_silent", "d5_rule30_ever"]

marg = defs.agg(F.count(F.lit(1)).alias("accounts"),
                *[F.sum(d).alias(d) for d in DEFS]).toPandas().iloc[0]
marg_pdf = pd.DataFrame({"accounts_flagged": marg[DEFS].astype("int64")})
marg_pdf["share"] = marg_pdf["accounts_flagged"] / int(marg["accounts"])

# Pairwise agreement against the flag
pairs = []
for d in DEFS:
    if d == "d1_status_C":
        continue
    r = defs.agg(
        F.sum(((F.col("d1_status_C") == 1) & (F.col(d) == 1)).cast("int")).alias("both"),
        F.sum(((F.col("d1_status_C") == 1) & (F.col(d) == 0)).cast("int")).alias("flag_only"),
        F.sum(((F.col("d1_status_C") == 0) & (F.col(d) == 1)).cast("int")).alias("def_only"),
        F.sum(((F.col("d1_status_C") == 0) & (F.col(d) == 0)).cast("int")).alias("neither"),
    ).toPandas().iloc[0]
    both, fo, do = int(r["both"]), int(r["flag_only"]), int(r["def_only"])
    pairs.append({"definition": d, "both": both, "flag_only": fo, "def_only": do,
                  "neither": int(r["neither"]),
                  "recall_of_flag": both / max(both + fo, 1),
                  "precision_vs_flag": both / max(both + do, 1)})
pair_pdf = pd.DataFrame(pairs)

profile = (defs.groupBy(*DEFS).count().orderBy(F.desc("count")).limit(15).toPandas())

# Timing: closure month by flag vs by silence
timing = (defs.filter((F.col("d2_closed_dt") == 1) & (F.col("d3_went_silent") == 1))
          .withColumn("gap_days", F.datediff("last_dt", "closed_dt_max"))
          .agg(F.count(F.lit(1)).alias("n"),
               F.expr("percentile_approx(gap_days, array(0.05,0.25,0.5,0.75,0.95))").alias("p"))
          .toPandas().iloc[0])

H("§6 · Closure definitions")
T(kv({"definition window cutoff for 'went silent'": f"last row before {gone_cut}",
      "zero-balance tolerance": f"|balance| <= {ZERO_TOL:,.0f}",
      "30%-rule baseline floor": f"prior-6 mean > ${MIN_BASE_BAL:,.0f}"}), "Parameters")
T(marg_pdf, "How many accounts each definition flags")
T(pair_pdf, "Every definition scored against d1 (the acct_status flag)")
T(profile, "Definition co-occurrence profile (top 15 combinations)")
T(kv({"accounts flagged by both closed_dt and silence": f"{int(timing['n']):,}",
      "last_dt - closed_dt (p05/p25/p50/p75/p95, days)": timing["p"]}),
  "Timing gap between the two closure records")

ans("closure_definition_agreement",
    "; ".join(f"{r.definition}: recall {r.recall_of_flag:.0%} / precision {r.precision_vs_flag:.0%}"
              for r in pair_pdf.itertuples()))

## §7 · Payments profile — direction, rails, coverage, duplicate legs

In [ ]:
direction = (F.when(F.col("mdm_pays").isNotNull() & F.col("mdm_recv").isNotNull(), "internal_c2c")
             .when(F.col("mdm_pays").isNull() & F.col("mdm_recv").isNotNull(), "inbound_from_cpty")
             .when(F.col("mdm_pays").isNotNull() & F.col("mdm_recv").isNull(), "outbound_to_cpty")
             .otherwise("both_null_ANOMALY"))

payd = pay.withColumn("direction", direction).persist(StorageLevel.MEMORY_AND_DISK)
payd.createOrReplaceTempView("v_payd")

dir_pdf = (payd.groupBy("direction").agg(
    F.count(F.lit(1)).alias("rows"),
    F.sum("trans_amt").alias("amount"),
    F.avg(F.col("cpty_acct").isNull().cast("double")).alias("null_cpty_acct"),
    F.avg(F.col("cpty_name").isNull().cast("double")).alias("null_cpty_name"),
    F.avg(F.col("cpty_fi").isNull().cast("double")).alias("null_cpty_fi"),
    F.avg(F.col("acct_pays").isNull().cast("double")).alias("null_acct_pays"),
    F.avg(F.col("acct_recv").isNull().cast("double")).alias("null_acct_recv"),
).orderBy(F.desc("rows")).toPandas())
dir_pdf["row_share"] = dir_pdf["rows"] / dir_pdf["rows"].sum()

# Q7: is there always a PNC account where there is an mdm_id?
acct_cov = payd.agg(
    F.avg(F.when(F.col("mdm_pays").isNotNull(),
                 F.col("acct_pays").isNull().cast("double"))).alias("pays_mdm_no_acct"),
    F.avg(F.when(F.col("mdm_recv").isNotNull(),
                 F.col("acct_recv").isNull().cast("double"))).alias("recv_mdm_no_acct"),
    F.avg(F.when(F.col("acct_pays").isNotNull(),
                 F.col("mdm_pays").isNull().cast("double"))).alias("pays_acct_no_mdm"),
    F.avg(F.when(F.col("acct_recv").isNotNull(),
                 F.col("mdm_recv").isNull().cast("double"))).alias("recv_acct_no_mdm"),
).toPandas().iloc[0]

rail_pdf = (payd.groupBy("rail", "category").agg(
    F.count(F.lit(1)).alias("rows"), F.sum("trans_amt").alias("amount"))
    .orderBy(F.desc("rows")).limit(40).toPandas())
rail_pdf["row_share"] = rail_pdf["rows"] / rail_pdf["rows"].sum()

amt_pdf = payd.agg(
    F.avg((F.col("trans_amt") < 0).cast("double")).alias("share_negative"),
    F.avg((F.col("trans_amt") == 0).cast("double")).alias("share_zero"),
    F.avg(F.col("trans_amt").isNull().cast("double")).alias("share_null"),
    F.expr("percentile_approx(trans_amt, array(0.01,0.25,0.5,0.75,0.99,0.999))").alias("p"),
).toPandas().iloc[0]

# Duplicate economic legs: same day, same amount, same two accounts, same rail
econ = (payd.groupBy("trans_dt", "trans_amt", "acct_pays", "acct_recv", "rail")
        .agg(F.count(F.lit(1)).alias("n")))
econ_dup = econ.filter("n > 1").agg(
    F.count(F.lit(1)).alias("dup_keys"), F.sum("n").alias("rows_in_dup_keys"),
    F.max("n").alias("max_per_key")).toPandas().iloc[0]
econ_tot = econ.agg(F.count(F.lit(1)).alias("keys"), F.sum("n").alias("rows")).toPandas().iloc[0]

# Same-name outflow feasibility (exact match, per your instruction)
same_name = (payd.filter(F.col("direction") == "outbound_to_cpty")
             .agg(F.count(F.lit(1)).alias("outbound_rows"),
                  F.avg((F.upper(F.trim("cpty_name")) == F.upper(F.trim("name_pays")))
                        .cast("double")).alias("exact_name_match_share"),
                  F.avg(F.col("cpty_fi").isNotNull().cast("double")).alias("fi_named_share"))
             .toPandas().iloc[0])

top_fi = (payd.filter(F.col("cpty_fi").isNotNull())
          .groupBy("cpty_fi").agg(F.count(F.lit(1)).alias("rows"),
                                  F.sum("trans_amt").alias("amount"))
          .orderBy(F.desc("rows")).limit(15).toPandas())

H("§7 · Payments profile")
T(dir_pdf, "Direction split, with null-share of each counterparty / account column")
T(kv({
    "mdm_id_pays present but pnc_dep_acct_pays NULL": f"{(acct_cov['pays_mdm_no_acct'] or 0):.3%}",
    "mdm_id_receives present but pnc_dep_acct_receives NULL": f"{(acct_cov['recv_mdm_no_acct'] or 0):.3%}",
    "acct_pays present but mdm_id_pays NULL": f"{(acct_cov['pays_acct_no_mdm'] or 0):.3%}",
    "acct_recv present but mdm_id_receives NULL": f"{(acct_cov['recv_acct_no_mdm'] or 0):.3%}",
}), "Q7 — is there always an account where there is an mdm_id?")
T(rail_pdf, "payment_rail x category")
T(pd.DataFrame({"pctile": ["p01", "p25", "p50", "p75", "p99", "p999"], "trans_amt": amt_pdf["p"]}),
  "Transaction amount")
T(kv({"negative amounts": f"{amt_pdf['share_negative']:.4%}",
      "zero amounts": f"{amt_pdf['share_zero']:.4%}",
      "null amounts": f"{amt_pdf['share_null']:.4%}"}))
T(kv({
    "distinct (date, amount, acct_pays, acct_recv, rail) keys": f"{int(econ_tot['keys']):,}",
    "keys carrying more than one row": f"{int(econ_dup['dup_keys'] or 0):,}",
    "share of keys duplicated": f"{(econ_dup['dup_keys'] or 0) / max(int(econ_tot['keys']), 1):.4%}",
    "max rows on one key": int(econ_dup["max_per_key"] or 0),
}), "Duplicate-leg check (distinct trans_id, but is the same payment written twice?)")
T(kv({
    "outbound rows": f"{int(same_name['outbound_rows']):,}",
    "cpty_name EXACTLY matches the paying customer's own name":
        f"{same_name['exact_name_match_share']:.4%}",
    "outbound rows with a named counterparty bank (cpty_fin_entity_name)":
        f"{same_name['fi_named_share']:.1%}",
}), "Group A feasibility — same-name outflow and fi_destination_flag")
T(top_fi, "Top counterparty financial institutions")

ans("payment_direction_split",
    "; ".join(f"{r.direction} {r.row_share:.1%}" for r in dir_pdf.itertuples()))
ans("mdm_without_account",
    f"pays {(acct_cov['pays_mdm_no_acct'] or 0):.2%} / recv {(acct_cov['recv_mdm_no_acct'] or 0):.2%}")
ans("duplicate_economic_legs",
    f"{(econ_dup['dup_keys'] or 0) / max(int(econ_tot['keys']), 1):.3%} of (date, amt, accts, rail) keys")
ans("fi_name_coverage_outbound", f"{same_name['fi_named_share']:.1%}")

## §8 · Join coverage: payments ↔ deposits, and `mdm_id` ↔ `cust_pwr_id`

In [ ]:
dep_accts = acct.select("acct", "cust_pwr_id").persist(StorageLevel.MEMORY_AND_DISK)

len_pay = (payd.select(F.explode(F.array(
    F.struct(F.lit("pay.acct_pays").alias("side"), F.length("acct_pays").alias("len")),
    F.struct(F.lit("pay.acct_recv").alias("side"), F.length("acct_recv").alias("len")))).alias("e"))
    .select("e.side", "e.len").filter(F.col("len").isNotNull())
    .groupBy("side", "len").count().orderBy("side", "len").toPandas())
len_dep = (dep_accts.groupBy(F.length("acct").alias("len")).count()
           .orderBy("len").toPandas().assign(side="dep.acct_full_acct_id"))

legs = (payd
        .select(F.explode(F.array(
            F.struct(F.lit("out").alias("leg"), F.col("acct_pays").alias("pnc_acct"),
                     F.col("mdm_pays").alias("mdm"), F.col("trans_dt"), F.col("trans_amt"),
                     F.col("cpty_acct")),
            F.struct(F.lit("in").alias("leg"), F.col("acct_recv").alias("pnc_acct"),
                     F.col("mdm_recv").alias("mdm"), F.col("trans_dt"), F.col("trans_amt"),
                     F.col("cpty_acct")))).alias("e"))
        .select("e.*").filter(F.col("pnc_acct").isNotNull()))

legs_j = legs.join(dep_accts.withColumnRenamed("acct", "pnc_acct"), "pnc_acct", "left")

cov_pdf = (legs_j.groupBy("leg").agg(
    F.count(F.lit(1)).alias("legs"),
    F.sum("trans_amt").alias("amount"),
    F.avg(F.col("cust_pwr_id").isNotNull().cast("double")).alias("share_joined_to_deposits"),
    F.sum(F.when(F.col("cust_pwr_id").isNotNull(), F.col("trans_amt")).otherwise(0.0))
    .alias("amount_joined"),
).toPandas())
cov_pdf["amount_coverage"] = cov_pdf["amount_joined"] / cov_pdf["amount"]

cov_month = (legs_j.groupBy(F.date_format("trans_dt", "yyyy-MM").alias("month"))
             .agg(F.avg(F.col("cust_pwr_id").isNotNull().cast("double")).alias("leg_coverage"))
             .orderBy("month").toPandas())

# mdm_id <-> cust_pwr_id cardinality, via joined legs
pairs_mc = (legs_j.filter(F.col("mdm").isNotNull() & F.col("cust_pwr_id").isNotNull())
            .select("mdm", "cust_pwr_id").distinct().persist())
card = pairs_mc.groupBy("mdm").agg(F.count(F.lit(1)).alias("n")).agg(
    F.count(F.lit(1)).alias("distinct_mdm"),
    F.avg((F.col("n") == 1).cast("double")).alias("share_mdm_with_one_cust_pwr"),
    F.max("n").alias("max_cust_pwr_per_mdm")).toPandas().iloc[0]
card2 = pairs_mc.groupBy("cust_pwr_id").agg(F.count(F.lit(1)).alias("n")).agg(
    F.count(F.lit(1)).alias("distinct_cust_pwr"),
    F.avg((F.col("n") == 1).cast("double")).alias("share_cust_pwr_with_one_mdm"),
    F.max("n").alias("max_mdm_per_cust_pwr")).toPandas().iloc[0]

# The anchor set and its counterparty reach
anchor = (legs_j.filter(F.col("cust_pwr_id").isNotNull())
          .agg(F.approx_count_distinct("pnc_acct", 0.01).alias("anchor_accounts"),
               F.approx_count_distinct("cust_pwr_id", 0.01).alias("anchor_customers"),
               F.approx_count_distinct("cpty_acct", 0.01).alias("connected_counterparties"))
          .toPandas().iloc[0])

H("§8 · Join coverage and identifier cardinality")
T(pd.concat([len_pay[["side", "len", "count"]],
             len_dep[["side", "len", "count"]]], ignore_index=True),
  "Account-id string length, both tables (a mismatch here is a padding problem)")
T(cov_pdf, "Payment legs with a PNC account: share that join to the deposit table")
T(cov_month, "Leg-level join coverage by month")
T(kv({
    "distinct mdm_id (joined)": f"{int(card['distinct_mdm']):,}",
    "mdm_id mapping to exactly one cust_pwr_id": f"{card['share_mdm_with_one_cust_pwr']:.3%}",
    "max cust_pwr_id per mdm_id": int(card["max_cust_pwr_per_mdm"]),
    "distinct cust_pwr_id (joined)": f"{int(card2['distinct_cust_pwr']):,}",
    "cust_pwr_id mapping to exactly one mdm_id": f"{card2['share_cust_pwr_with_one_mdm']:.3%}",
    "max mdm_id per cust_pwr_id": int(card2["max_mdm_per_cust_pwr"]),
}), "mdm_id <-> cust_pwr_id — is it really one-to-one?")
T(kv({
    "anchor accounts (DDA/MMDA, deposit + payment)": f"~{int(anchor['anchor_accounts']):,}",
    "anchor customers": f"~{int(anchor['anchor_customers']):,}",
    "counterparties connected to the anchor set": f"~{int(anchor['connected_counterparties']):,}",
}), "The extraction anchor")

ans("leg_join_coverage",
    "; ".join(f"{r.leg} {r.share_joined_to_deposits:.1%} legs / {r.amount_coverage:.1%} $"
              for r in cov_pdf.itertuples()))
ans("mdm_cust_pwr_one_to_one",
    f"mdm->cust_pwr {card['share_mdm_with_one_cust_pwr']:.2%} 1:1, "
    f"cust_pwr->mdm {card2['share_cust_pwr_with_one_mdm']:.2%} 1:1")
ans("anchor_accounts", f"~{int(anchor['anchor_accounts']):,}")
ans("connected_counterparties", f"~{int(anchor['connected_counterparties']):,}")

## §9 · Attrition feasibility

The three numbers that decide whether the study can run at all:
**closures per month** (a cliff means the panel is truncated, not that
departures stopped), **the decline shape** relative to closure, and the
**episode count** with enough history to model.

In [ ]:
closed = (defs.filter(F.col("d2_closed_dt") == 1)
          .select("acct", "cust_pwr_id",
                  F.date_format("closed_dt_max", "yyyy-MM").alias("close_month"),
                  (F.year("closed_dt_max") * 12 + F.month("closed_dt_max")).alias("close_idx"),
                  "n_months"))

per_month = (closed.groupBy("close_month").count().orderBy("close_month").toPandas()
             .rename(columns={"count": "accounts_closed"}))
open_by_month = (mp.groupBy("month").agg(F.countDistinct("acct").alias("accounts_reporting"))
                 .orderBy("month").toPandas())
rate = open_by_month.merge(per_month, left_on="month", right_on="close_month", how="left")
rate["accounts_closed"] = rate["accounts_closed"].fillna(0).astype(int)
rate["closure_rate"] = rate["accounts_closed"] / rate["accounts_reporting"]

# Decline shape, indexed to the closure month
traj = (mp.join(closed.select("acct", "close_idx"), "acct", "inner")
        .withColumn("rel_m", F.col("month_idx") - F.col("close_idx"))
        .filter(F.col("rel_m").between(-12, 0)))
base = (traj.filter(F.col("rel_m") == -12).select("acct", F.col("bal_eom").alias("base_bal"))
        .filter(F.col("base_bal") > MIN_BASE_BAL))
traj_idx = (traj.join(base, "acct", "inner")
            .withColumn("idx", 100.0 * F.col("bal_eom") / F.col("base_bal"))
            .groupBy("rel_m")
            .agg(F.count(F.lit(1)).alias("n"),
                 F.expr("percentile_approx(idx, 0.5)").alias("median_index"),
                 F.expr("percentile_approx(idx, 0.25)").alias("p25"),
                 F.expr("percentile_approx(idx, 0.75)").alias("p75"))
            .orderBy("rel_m").toPandas())

# Control: accounts that never closed and never went silent, indexed to the end
# of the window so the T-12..T0 span is the same length as the cases' span.
ctrl_anchor = mp.agg(F.max("month_idx")).collect()[0][0]
ctrl_src = mp.join(defs.filter((F.col("d2_closed_dt") == 0) & (F.col("d3_went_silent") == 0))
                   .select("acct"), "acct", "inner")
ctrl = ctrl_src.withColumn("rel_m", F.col("month_idx") - F.lit(ctrl_anchor))
ctrl_base = (ctrl.filter(F.col("rel_m") == -12).select("acct", F.col("bal_eom").alias("base_bal"))
             .filter(F.col("base_bal") > MIN_BASE_BAL))
ctrl_idx = (ctrl.filter(F.col("rel_m").between(-12, 0)).join(ctrl_base, "acct", "inner")
            .withColumn("idx", 100.0 * F.col("bal_eom") / F.col("base_bal"))
            .groupBy("rel_m").agg(F.expr("percentile_approx(idx, 0.5)").alias("control_median"))
            .orderBy("rel_m").toPandas())

shape = traj_idx.merge(ctrl_idx, on="rel_m", how="left")

episodes = (closed.filter(F.col("n_months") >= 12)
            .agg(F.count(F.lit(1)).alias("episodes_ge_12m_history"),
                 F.approx_count_distinct("cust_pwr_id", 0.01).alias("distinct_customers"))
            .toPandas().iloc[0])

H("§9 · Attrition feasibility")
T(rate[["month", "accounts_reporting", "accounts_closed", "closure_rate"]],
  "Closures per month — a collapse toward the start means the panel is truncated, "
  "not that departures stopped")
T(shape, "Median month-end balance indexed to 100 at T-12, cases vs. non-closing controls")
T(kv({
    "closed accounts with >=12 months of history": f"{int(episodes['episodes_ge_12m_history']):,}",
    "distinct customers behind them": f"~{int(episodes['distinct_customers']):,}",
    "usable episode months (12m burn-in)":
        f"{DATE_START[:7]} +12 .. {DATE_END[:7]}",
}), "Episode supply")

try:
    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(1, 2, figsize=(13, 4))
    ax[0].bar(rate["month"], rate["closure_rate"])
    ax[0].set_title("Monthly closure rate"); ax[0].tick_params(axis="x", rotation=90, labelsize=7)
    ax[1].plot(shape["rel_m"], shape["median_index"], marker="o", label="closing accounts")
    if "control_median" in shape:
        ax[1].plot(shape["rel_m"], shape["control_median"], marker="s", label="controls")
    ax[1].axhline(100, ls="--", lw=0.8, c="grey")
    ax[1].set_title("Balance indexed to 100 at T-12"); ax[1].set_xlabel("months to closure")
    ax[1].legend(); plt.tight_layout(); plt.show()
except Exception as e:
    print(f"(plot skipped: {e})")

ans("closures_per_month_min_max",
    f"{int(rate['accounts_closed'].min()):,} .. {int(rate['accounts_closed'].max()):,}")
ans("episodes_ge_12m", f"{int(episodes['episodes_ge_12m_history']):,}")

## §10 · Scoreboard — every answer in one place

In [ ]:
H("§10 · Scoreboard")
T(kv(ANSWERS, "answer"), "Findings from this run")
print("\nOpen items still requiring a human decision:")
for line in [
    "1. Confirm the DDA/MMDA family list in §2 if auto-detect mis-classified anything.",
    "2. If §4 says CURRENT-STATE, acct_status is an outcome label only — the as-of state "
    "must be reconstructed from closed_dt or from row disappearance.",
    "3. Pick the closure definition to carry forward from §6, and record why.",
    "4. If §5 shows amb1 is a prior-period figure, the 30% rule as currently run is "
    "operating on a one-month-lagged input — worth telling Ops.",
]:
    print("  " + line)

In [ ]:
for _df in (dep, mp, acct, defs, payd, key_ct, dep_accts, pairs_mc):
    try:
        _df.unpersist()
    except Exception:
        pass
print("caches released")